# Google Places Aggregate POI features — WA 49-site extraction

This notebook documents the extraction of structured destination-context features for the 49 pedestrian observation sites recorded in `data/processed/model_dataset.xlsx`. The analysis uses the **Google Places Aggregate API** and is limited to `INSIGHT_COUNT`, so the resulting dataset contains aggregate quantities rather than place names, addresses, reviews, photos or Place IDs.

Each `site_id` is represented by a fixed **500 m radius**, the spatial specification retained after the pilot analysis. The surrounding destination environment is represented through prepared food/drink, grocery/convenience/market shopping, major shopping destinations and other retail. This separation preserves the distinction between the presence of a large destination, such as a shopping mall, and the number of smaller establishments.

The resulting candidate fields are summarised below.

| Feature family | Output field | Interpretation |
| --- | --- | --- |
| Prepared food and drink | `google_food_drink_count_500m` | Operational places whose primary type matches `restaurant`, `cafe`, `bakery` or `bar`, including official descendants |
| Grocery, convenience and market shopping | `google_grocery_convenience_market_count_500m` | Operational places whose primary type matches `supermarket`, `grocery_store`, `convenience_store`, `food_store` or `market`, including official descendants |
| Major shopping | `google_major_shopping_present_500m` | Whether at least one major shopping destination exists within 500 m |
| Other retail | `google_other_retail_count_500m` | Number of other operational retail destinations within 500 m |

`OPERATING_STATUS_OPERATIONAL` means that a place is operational and opens during its defined hours. It is **not** a real-time `openNow` test: the request contains no time-of-day or current-opening-hours filter. A restaurant is therefore retained when it is closed tonight or on its normal weekly rest day, provided Google classifies it as operational. Running the same query at night rather than during the day does not change the rule; counts can still change later if Google's place data or operating status is updated.

Official sources: [Places Aggregate overview](https://developers.google.com/maps/documentation/places-aggregate/overview), [request parameters](https://developers.google.com/maps/documentation/places-aggregate/request-parameters), [type-filter logic](https://developers.google.com/maps/documentation/places-aggregate/more-on-type-filters), and [Place Types Table A](https://developers.google.com/maps/documentation/places/web-service/place-types#table-a).

In [ ]:
from __future__ import annotations

import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from openpyxl.styles import Alignment, Font, PatternFill


def find_project_root() -> Path:
    # Works when Jupyter starts either in the repository root or a subfolder.
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "processed" / "model_dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the 03_Project repository containing "
        "data/processed/model_dataset.xlsx."
    )


PROJECT_ROOT = find_project_root()
MODEL_DATASET_FILE = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "02_feature_extraction"
FEATURE_FILE = (
    OUTPUT_DIR / "02_08_haoqi_google_places_aggregate_poi_features.xlsx"
)

load_dotenv(PROJECT_ROOT / ".env")

RADIUS_M = 500
MAX_WORKERS = 4
EXPECTED_SITE_COUNT = 49

print("Project root :", PROJECT_ROOT)
print("Model input  :", MODEL_DATASET_FILE.relative_to(PROJECT_ROOT))
print("Radius (m)   :", RADIUS_M)
print("Excel output :", FEATURE_FILE.relative_to(PROJECT_ROOT))
print("API key set  :", bool(os.getenv("GOOGLE_MAPS_API_KEY")))

## 1. Google Place type definitions used in the analysis

Google's official material provides the canonical Table A type identifiers and the rules governing type filters, but it does not define the four analytical families used in this project. The family labels therefore describe the project's aggregation design, whereas every identifier supplied to the API is an official Google Place type.

Classification is based on `includedPrimaryTypes`. A place is consequently represented by its **single primary function**, rather than being included solely because it carries a secondary tag. Under Google's hierarchical type system, descendants of each specified type are included automatically and the values within a single list are evaluated with OR logic. The request roots used in the analysis are shown below.

| Project family | Official `includedPrimaryTypes` | Official `excludedPrimaryTypes` | Exact API interpretation |
| --- | --- | --- | --- |
| Prepared food and drink | `restaurant`, `cafe`, `bakery`, `bar` | none | Primary type is one of these official types or an official descendant, such as a cuisine-specific restaurant under `restaurant` |
| Grocery, convenience and market shopping | `supermarket`, `grocery_store`, `convenience_store`, `food_store`, `market` | none | Primary function is one of these five official types or a descendant; `market` is retained explicitly rather than silently described as grocery |
| Major shopping destination | `shopping_mall`, `department_store`, `hypermarket`, `warehouse_store` | none | Primary function is one of these four official large-format shopping types or a descendant |
| Other retail | `store` | all 13 roots above | Primary type is `store` or a descendant, after removing the prepared-food, grocery/convenience/market and major-shopping roots and their descendants |

The first feature therefore represents the stated prepared-food/drink subset rather than every type listed under Google's broader “Food and Drink” heading. The second label retains `market` explicitly because that official type is broader than grocery alone. Major shopping is represented as a presence indicator because a mall or hypermarket is not commensurate with one small shop. No overall shopping total or duplicate density measure is introduced.

In [ ]:
FOOD_DRINK_PRIMARY_TYPES = ["restaurant", "cafe", "bakery", "bar"]
GROCERY_CONVENIENCE_MARKET_PRIMARY_TYPES = [
    "supermarket",
    "grocery_store",
    "convenience_store",
    "food_store",
    "market",
]
MAJOR_SHOPPING_PRIMARY_TYPES = [
    "shopping_mall",
    "department_store",
    "hypermarket",
    "warehouse_store",
]
OTHER_RETAIL_EXCLUSIONS = sorted(
    set(
        FOOD_DRINK_PRIMARY_TYPES
        + GROCERY_CONVENIENCE_MARKET_PRIMARY_TYPES
        + MAJOR_SHOPPING_PRIMARY_TYPES
    )
)

CATEGORY_CONFIG = {
    "food_drink": {
        "label": "Food and drink destinations",
        "type_filter": {"includedPrimaryTypes": FOOD_DRINK_PRIMARY_TYPES},
        "output_kind": "count",
    },
    "grocery_convenience_market": {
        "label": "Grocery, convenience and market shopping",
        "type_filter": {
            "includedPrimaryTypes": GROCERY_CONVENIENCE_MARKET_PRIMARY_TYPES
        },
        "output_kind": "count",
    },
    "major_shopping": {
        "label": "Major shopping destinations",
        "type_filter": {
            "includedPrimaryTypes": MAJOR_SHOPPING_PRIMARY_TYPES
        },
        "output_kind": "presence",
    },
    "other_retail": {
        "label": "Other retail destinations",
        "type_filter": {
            "includedPrimaryTypes": ["store"],
            "excludedPrimaryTypes": OTHER_RETAIL_EXCLUSIONS,
        },
        "output_kind": "count",
    },
}

type_rows = []
for category, config in CATEGORY_CONFIG.items():
    type_rows.append(
        {
            "category": category,
            "output_kind": config["output_kind"],
            "included_primary_types": ", ".join(
                config["type_filter"].get("includedPrimaryTypes", [])
            ),
            "excluded_primary_types": ", ".join(
                config["type_filter"].get("excludedPrimaryTypes", [])
            ),
        }
    )
display(pd.DataFrame(type_rows))

## 2. Observation sites and spatial unit

The combination of `road_no + slk_km` provides a linear reference to a point on a Main Roads WA road and does not itself define a segment length. Because the current coordinates were derived through AusRAP segment interpolation, the survey and prediction unit in this analysis is treated as a point.

For a future client batch supplied as SLK rather than coordinates, the corresponding SLK point would be converted to WGS84 before the same spatial definition is applied. Prediction over an entire `[start_slk, end_slk]` interval would constitute a different spatial problem, requiring road geometry and either a corridor representation or multiple sample points; a circle around one midpoint would not describe the full interval.

The common 500 m radius gives the 49 sites a consistent basis for comparison. It represents environmental context around the observation point and does not redefine that point as a one-kilometre road segment.

In [ ]:
INPUT_COLUMNS = [
    "site_id",
    "road_no",
    "slk_km",
    "latitude",
    "longitude",
    "coordinate_crs",
    "coordinate_method",
    "qa_status",
]

model_data = pd.read_excel(
    MODEL_DATASET_FILE,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "road_no": "string"},
)

missing_columns = set(INPUT_COLUMNS).difference(model_data.columns)
if missing_columns:
    raise ValueError(f"Model_Data is missing fields: {sorted(missing_columns)}")

sites = model_data[INPUT_COLUMNS].copy()
sites["site_id"] = sites["site_id"].str.strip()
sites["latitude"] = pd.to_numeric(sites["latitude"], errors="coerce")
sites["longitude"] = pd.to_numeric(sites["longitude"], errors="coerce")

if len(sites) != EXPECTED_SITE_COUNT:
    raise ValueError(f"Expected {EXPECTED_SITE_COUNT} Model_Data rows, found {len(sites)}.")
if sites["site_id"].isna().any() or sites["site_id"].eq("").any():
    raise ValueError("site_id cannot be missing or blank.")
if sites["site_id"].duplicated().any():
    raise ValueError("site_id must be unique.")
if sites[["latitude", "longitude"]].isna().any().any():
    raise ValueError("All sites require numeric latitude and longitude.")
if not sites["coordinate_crs"].eq("EPSG:4326").all():
    raise ValueError("Every site must use coordinate_crs == 'EPSG:4326'.")
if not sites["latitude"].between(-36, -13).all():
    raise ValueError("Latitude values fall outside the expected WA range.")
if not sites["longitude"].between(112, 130).all():
    raise ValueError("Longitude values fall outside the expected WA range.")

expected_requests = len(sites) * len(CATEGORY_CONFIG)
print(f"Validated Model_Data sites : {len(sites)}")
print(f"Expected count requests    : {expected_requests}")
display(
    sites[["site_id", "road_no", "slk_km", "coordinate_method"]]
    .head(10)
)

## 3. Aggregate request design

Authentication is provided through `GOOGLE_MAPS_API_KEY` in `.env`, while the key itself remains outside the notebook output. Each request combines one circular area, one fixed type filter and `OPERATING_STATUS_OPERATIONAL`. The response is parsed in memory to obtain the aggregate count and the raw JSON is not retained.

The request logic allows up to three retries for transient 429 and 5xx responses. Authentication failures and invalid requests are treated as terminal for the affected request. This distinction preserves the meaning of zero as a successful count: request failures remain missing rather than being converted to zero.

In [ ]:
API_URL = "https://areainsights.googleapis.com/v1:computeInsights"
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "GOOGLE_MAPS_API_KEY is missing. Add it to the project .env file."
    )

RETRY_DELAYS_SECONDS = (1, 3, 8)


class PlacesAggregateError(RuntimeError):
    pass


def safe_api_message(response: requests.Response) -> str:
    try:
        message = response.json().get("error", {}).get("message", "")
    except ValueError:
        message = ""
    return (message or f"HTTP {response.status_code}")[:300]


def request_poi_count(
    session: requests.Session,
    latitude: float,
    longitude: float,
    radius_m: int,
    type_filter: dict,
) -> int:
    payload = {
        "insights": ["INSIGHT_COUNT"],
        "filter": {
            "locationFilter": {
                "circle": {
                    "latLng": {
                        "latitude": float(latitude),
                        "longitude": float(longitude),
                    },
                    "radius": int(radius_m),
                }
            },
            "typeFilter": type_filter,
            "operatingStatus": ["OPERATING_STATUS_OPERATIONAL"],
        },
    }

    for attempt in range(len(RETRY_DELAYS_SECONDS) + 1):
        try:
            response = session.post(
                API_URL,
                headers={
                    "X-Goog-Api-Key": API_KEY,
                    "Content-Type": "application/json",
                },
                json=payload,
                timeout=(10, 40),
            )
        except requests.RequestException as error:
            if attempt < len(RETRY_DELAYS_SECONDS):
                time.sleep(RETRY_DELAYS_SECONDS[attempt])
                continue
            raise PlacesAggregateError(type(error).__name__) from error

        if response.status_code == 429 or 500 <= response.status_code < 600:
            if attempt < len(RETRY_DELAYS_SECONDS):
                time.sleep(RETRY_DELAYS_SECONDS[attempt])
                continue

        if not response.ok:
            raise PlacesAggregateError(safe_api_message(response))

        try:
            count = int(response.json()["count"])
        except (KeyError, TypeError, ValueError) as error:
            raise PlacesAggregateError("Response did not contain a valid count.") from error
        if count < 0:
            raise PlacesAggregateError("API returned a negative count.")
        return count

    raise PlacesAggregateError("Request failed after retries.")


def output_column(category: str, output_kind: str) -> str:
    suffix = "present" if output_kind == "presence" else "count"
    return f"google_{category}_{suffix}_{RADIUS_M}m"


def extract_one_site(site: pd.Series) -> dict:
    result = {
        "site_id": site["site_id"],
        "google_places_data_status": "not_run",
        "google_places_error_message": pd.NA,
    }
    errors = []
    succeeded = 0

    with requests.Session() as session:
        for category, config in CATEGORY_CONFIG.items():
            column = output_column(category, config["output_kind"])
            try:
                count = request_poi_count(
                    session=session,
                    latitude=site["latitude"],
                    longitude=site["longitude"],
                    radius_m=RADIUS_M,
                    type_filter=config["type_filter"],
                )
                result[column] = (
                    bool(count > 0)
                    if config["output_kind"] == "presence"
                    else count
                )
                succeeded += 1
            except PlacesAggregateError as error:
                result[column] = pd.NA
                errors.append(f"{category}_{RADIUS_M}m: {error}")

    expected = len(CATEGORY_CONFIG)
    if succeeded == expected:
        result["google_places_data_status"] = "complete"
    elif succeeded == 0:
        result["google_places_data_status"] = "request_failed"
    else:
        result["google_places_data_status"] = "partial_failure"
    if errors:
        result["google_places_error_message"] = " | ".join(errors)[:1000]
    return result

## 4. Batch extraction across the 49 sites

The 49 sites are processed independently across four worker threads. For each site, one worker completes the four 500 m category requests within a dedicated HTTP session. A failure associated with one site is recorded locally and does not interrupt the remaining observations.

During extraction, the notebook reports progress and the resulting status totals. The complete site-level feature table is presented later, after its structure and values have been checked.

In [ ]:
site_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_site = {
        executor.submit(extract_one_site, row): row["site_id"]
        for _, row in sites.iterrows()
    }
    for completed, future in enumerate(as_completed(future_to_site), start=1):
        site_id = future_to_site[future]
        try:
            site_results.append(future.result())
        except Exception as error:
            site_results.append(
                {
                    "site_id": site_id,
                    "google_places_data_status": "request_failed",
                    "google_places_error_message": type(error).__name__,
                }
            )
        print(f"Completed {completed:>2}/{len(sites)} sites", end="\r")

print()
features = pd.DataFrame(site_results)
features = sites[["site_id"]].merge(
    features,
    on="site_id",
    how="left",
    validate="one_to_one",
)

count_columns = [
    output_column(category, config["output_kind"])
    for category, config in CATEGORY_CONFIG.items()
    if config["output_kind"] == "count"
]
presence_columns = [
    output_column(category, config["output_kind"])
    for category, config in CATEGORY_CONFIG.items()
    if config["output_kind"] == "presence"
]
for column in count_columns + presence_columns:
    if column not in features.columns:
        features[column] = pd.NA
for column in count_columns:
    features[column] = pd.to_numeric(
        features[column], errors="coerce"
    ).astype("Int64")
for column in presence_columns:
    features[column] = features[column].astype("boolean")

ordered_columns = [
    "site_id",
    "google_places_data_status",
    "google_places_error_message",
    "google_food_drink_count_500m",
    "google_grocery_convenience_market_count_500m",
    "google_major_shopping_present_500m",
    "google_other_retail_count_500m",
]
features = features[ordered_columns]

display(
    features["google_places_data_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .to_frame("sites")
)

## 5. Quality criteria for the 500 m feature table

A zero returned by a successful request is treated as observed environmental information. Missing values are associated only with a recorded request problem. The formal table is defined by one row per `site_id`, coverage of all 49 input sites, non-negative counts and complete feature values for every row marked `complete`.

An in-memory summary describes coverage, missingness and variation for quality assessment. It remains part of the notebook record and is not added as a second workbook sheet.

In [ ]:
if len(features) != EXPECTED_SITE_COUNT:
    raise AssertionError("Feature output must contain all 49 sites.")
if features["site_id"].duplicated().any():
    raise AssertionError("Feature output contains duplicate site_id values.")
if set(features["site_id"]) != set(sites["site_id"]):
    raise AssertionError("Feature site_id values do not match Model_Data.")

for column in count_columns:
    invalid = features[column].dropna().lt(0)
    if invalid.any():
        raise AssertionError(f"Negative values found in {column}.")

complete_mask = features["google_places_data_status"].eq("complete")
feature_columns = count_columns + presence_columns
if features.loc[complete_mask, feature_columns].isna().any().any():
    raise AssertionError("A complete site contains a missing feature value.")

summary_rows = []
for column in count_columns:
    values = features[column].dropna()
    summary_rows.append(
        {
            "feature": column,
            "feature_kind": "count",
            "available_sites": int(len(values)),
            "missing_sites": int(features[column].isna().sum()),
            "zero_or_false_sites": int(values.eq(0).sum()),
            "nonzero_or_true_sites": int(values.gt(0).sum()),
            "distinct_values": int(values.nunique()),
            "mean_count": float(values.mean()) if len(values) else np.nan,
            "true_rate": pd.NA,
            "median": float(values.median()) if len(values) else np.nan,
            "maximum": int(values.max()) if len(values) else pd.NA,
        }
    )
for column in presence_columns:
    values = features[column].dropna().astype(bool)
    summary_rows.append(
        {
            "feature": column,
            "feature_kind": "presence",
            "available_sites": int(len(values)),
            "missing_sites": int(features[column].isna().sum()),
            "zero_or_false_sites": int((~values).sum()),
            "nonzero_or_true_sites": int(values.sum()),
            "distinct_values": int(values.nunique()),
            "mean_count": pd.NA,
            "true_rate": float(values.mean()) if len(values) else np.nan,
            "median": pd.NA,
            "maximum": pd.NA,
        }
    )

feature_summary = pd.DataFrame(summary_rows)
print("Validation passed.")

## 6. Formal feature workbook

The formal output is stored at `06_outputs/02_feature_extraction/02_08_haoqi_google_places_aggregate_poi_features.xlsx`. Its single sheet, `Google_POI_Features`, contains 49 site-level rows and only the fields belonging to this independent feature source.

The workbook is treated as a short-lived research artefact. Its retention is limited to the period required for feature assessment, model training or prediction, with removal occurring within 30 days.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(FEATURE_FILE, engine="openpyxl") as writer:
    features.to_excel(
        writer, sheet_name="Google_POI_Features", index=False
    )

    header_fill = PatternFill("solid", fgColor="1F4E78")
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(horizontal="center", vertical="center")
        for cells in worksheet.columns:
            width = min(max(len(str(cell.value or "")) for cell in cells) + 2, 52)
            worksheet.column_dimensions[cells[0].column_letter].width = max(width, 12)

reloaded = pd.read_excel(
    FEATURE_FILE,
    sheet_name="Google_POI_Features",
    dtype={"site_id": "string"},
)
if len(reloaded) != EXPECTED_SITE_COUNT:
    raise AssertionError("Reloaded workbook must contain all 49 sites.")
if list(reloaded.columns) != ordered_columns:
    raise AssertionError("Reloaded workbook has unexpected feature columns.")
if reloaded["site_id"].duplicated().any():
    raise AssertionError("Reloaded workbook has duplicate site_id values.")

sheet_names = pd.ExcelFile(FEATURE_FILE).sheet_names
if sheet_names != ["Google_POI_Features"]:
    raise AssertionError(f"Unexpected workbook sheets: {sheet_names}")

print("Saved feature workbook:", FEATURE_FILE.relative_to(PROJECT_ROOT))
print(f"Rows saved: {len(reloaded)}")
print("Delete this short-lived workbook within 30 days.")

## 7. Features produced by the current run

The outputs below are calculated directly from the API responses obtained during the current run and therefore represent observed extraction results rather than hard-coded examples.

`features` contains the observed 500 m values for all 49 sites. `feature_summary` describes coverage, the balance between zero and non-zero observations, and the variation present in the four formal features. This summary remains within the notebook.

In [ ]:
completed_sites = int(
    features["google_places_data_status"].eq("complete").sum()
)
print(f"Queried sites   : {len(features)}")
print(f"Complete sites  : {completed_sites}")
print(f"Output workbook : {FEATURE_FILE.relative_to(PROJECT_ROOT)}")

display(features)
display(feature_summary)

## 8. Data lifecycle

The durable research record consists of the code, fixed type definitions, radius, non-sensitive quality summary, model evaluation and pedestrian predictions. Raw API responses are not retained, and the workflow does not create a historical Google POI database.

The feature workbook is removed within 30 days once it is no longer required for feature review, model training or prediction. A future client batch follows the same coordinate → aggregate count → prediction sequence, with the newly generated aggregate features again treated as short-lived data.